# Smart IoT Device Battery Management System

Software-only device health and battery monitoring dashboard.

**Parameters:** CPU, RAM, Disk, Temperature, Battery.

> In Google Colab, this monitors the Colab runtime. To monitor your actual laptop/PC, run the same monitoring logic locally on that device.

In [ ]:
# ============================================================
# SMART IoT DEVICE BMS - GOOGLE COLAB
# ============================================================

!pip -q install psutil plotly pandas numpy

import os
import time
import platform
import socket
from datetime import datetime

import numpy as np
import pandas as pd
import psutil
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import clear_output, display

history = []
MAX_HISTORY = 40


def get_temperature():
    try:
        temperatures = psutil.sensors_temperatures(fahrenheit=False)
        if not temperatures:
            return np.nan

        preferred = ("coretemp", "k10temp", "cpu_thermal", "cpu-thermal")
        for name in preferred:
            if name in temperatures:
                for entry in temperatures[name]:
                    if entry.current is not None:
                        return float(entry.current)

        for entries in temperatures.values():
            for entry in entries:
                if entry.current is not None:
                    return float(entry.current)
    except Exception:
        pass
    return np.nan


def get_battery():
    try:
        battery = psutil.sensors_battery()
        if battery is None:
            return np.nan, None
        return float(battery.percent), bool(battery.power_plugged)
    except Exception:
        return np.nan, None


def collect_data():
    cpu = psutil.cpu_percent(interval=0.5)
    ram = psutil.virtual_memory().percent
    disk = psutil.disk_usage(os.path.abspath(os.sep)).percent
    temp = get_temperature()
    battery, plugged = get_battery()
    uptime = (time.time() - psutil.boot_time()) / 3600

    return {
        "Time": datetime.now(),
        "CPU": cpu,
        "RAM": ram,
        "Disk": disk,
        "Temperature": temp,
        "Battery": battery,
        "Plugged": plugged,
        "Uptime": uptime
    }


def health_score(d):
    score = 100

    if d["CPU"] > 90:
        score -= 25
    elif d["CPU"] > 75:
        score -= 10

    if d["RAM"] > 90:
        score -= 20
    elif d["RAM"] > 75:
        score -= 10

    if d["Disk"] > 95:
        score -= 20
    elif d["Disk"] > 85:
        score -= 10

    if not np.isnan(d["Temperature"]):
        if d["Temperature"] > 85:
            score -= 30
        elif d["Temperature"] > 70:
            score -= 15

    if not np.isnan(d["Battery"]):
        if d["Battery"] < 10:
            score -= 20
        elif d["Battery"] < 20:
            score -= 10

    score = max(0, min(100, score))
    status = "NORMAL" if score >= 80 else "WARNING" if score >= 60 else "CRITICAL"
    return score, status


def alerts(d):
    a = []

    if d["CPU"] > 90: a.append("HIGH CPU")
    if d["RAM"] > 90: a.append("HIGH RAM")
    if d["Disk"] > 95: a.append("DISK ALMOST FULL")

    if not np.isnan(d["Temperature"]):
        if d["Temperature"] > 85: a.append("CRITICAL TEMPERATURE")
        elif d["Temperature"] > 70: a.append("HIGH TEMPERATURE")

    if not np.isnan(d["Battery"]):
        if d["Battery"] < 10: a.append("CRITICAL BATTERY")
        elif d["Battery"] < 20: a.append("LOW BATTERY")

    return a


def gauge(value, title, max_value=100, suffix="%"):
    if np.isnan(value):
        value = 0

    fig = go.Figure(go.Indicator(
        mode="gauge+number",
        value=value,
        title={"text": title},
        number={"suffix": suffix},
        gauge={"axis": {"range": [0, max_value]}}
    ))

    fig.update_layout(height=250, margin=dict(l=20,r=20,t=60,b=20))
    return fig


def dashboard():
    global history

    d = collect_data()
    history.append(d)
    history = history[-MAX_HISTORY:]

    df = pd.DataFrame(history)
    score, status = health_score(d)
    active_alerts = alerts(d)

    clear_output(wait=True)

    print("=" * 70)
    print("          SMART IoT DEVICE BMS - LIVE DASHBOARD")
    print("=" * 70)

    print(f"Device: {socket.gethostname()}")
    print(f"OS: {platform.system()} {platform.release()}")
    print()

    print(f"CPU Usage       : {d['CPU']:.2f} %")
    print(f"RAM Usage       : {d['RAM']:.2f} %")
    print(f"Disk Usage      : {d['Disk']:.2f} %")

    temp = "N/A" if np.isnan(d["Temperature"]) else f"{d['Temperature']:.2f} °C"
    bat = "N/A" if np.isnan(d["Battery"]) else f"{d['Battery']:.2f} %"

    print(f"Temperature     : {temp}")
    print(f"Battery         : {bat}")
    print(f"Health Score    : {score:.1f}/100")
    print(f"Status          : {status}")
    print(f"Uptime          : {d['Uptime']:.2f} hours")

    print("\nALERTS")
    if active_alerts:
        for x in active_alerts:
            print("⚠", x)
    else:
        print("✓ No abnormal conditions detected")

    display(gauge(d["CPU"], "CPU Usage"))
    display(gauge(d["RAM"], "RAM Usage"))
    display(gauge(d["Disk"], "Disk Usage"))

    if not np.isnan(d["Temperature"]):
        display(gauge(d["Temperature"], "Temperature", 100, " °C"))

    if not np.isnan(d["Battery"]):
        display(gauge(d["Battery"], "Battery Level"))

    if len(df) > 1:
        fig = make_subplots(
            rows=5, cols=1, shared_xaxes=True,
            vertical_spacing=0.04,
            subplot_titles=[
                "CPU Usage", "RAM Usage", "Disk Usage",
                "Temperature", "Battery Level"
            ]
        )

        fig.add_trace(go.Scatter(x=df["Time"], y=df["CPU"], mode="lines+markers"), row=1, col=1)
        fig.add_trace(go.Scatter(x=df["Time"], y=df["RAM"], mode="lines+markers"), row=2, col=1)
        fig.add_trace(go.Scatter(x=df["Time"], y=df["Disk"], mode="lines+markers"), row=3, col=1)
        fig.add_trace(go.Scatter(x=df["Time"], y=df["Temperature"], mode="lines+markers"), row=4, col=1)
        fig.add_trace(go.Scatter(x=df["Time"], y=df["Battery"], mode="lines+markers"), row=5, col=1)

        fig.update_layout(height=1100, title="Device BMS Historical Telemetry")
        display(fig)

    df.to_csv("device_bms_telemetry.csv", index=False)


# Run once first
dashboard()


## Continuous Monitoring

Run the following cell to refresh the dashboard every 5 seconds. Stop the cell with the Colab stop button.

In [ ]:
try:
    while True:
        dashboard()
        time.sleep(5)
except KeyboardInterrupt:
    print('Monitoring stopped.')
